# Multi-dataset ScDataLoader example

This notebook shows the full path-list workflow:

1. Start from a list of dataset paths.
2. Optionally convert AnnData-readable sources into same-name `.zarr.zip` stores.
3. Register all stores with `bank.register(map(launch, store_paths))`.
4. Build a torch-style dataset whose samples are `(file_id, cell_id)` pairs.
5. Use `ScDataLoader` to prefetch decoded cell batches from `ScDataBank`.
6. Collate the decoded cells into ordinary torch tensors for model code.


In [ ]:
from __future__ import annotations

from bisect import bisect_right
from pathlib import Path

import numpy as np
import torch

from scdata import AnnDataZarrZipConverter, ScDataBank, ScDataLoader, launch

## Input paths

`store_paths` should point to scdata-readable zarr directories or `.zarr.zip` files. If your input files are still `.h5ad`, `.loom`, `.mtx`, or standard AnnData zarr stores, set `convert_sources = True`; the converter writes same-name `.zarr.zip` outputs and returns those paths.

In [ ]:
raw_paths = [
    Path("/path/to/sample_a.zarr.zip"),
    Path("/path/to/sample_b.zarr.zip"),
]

convert_sources = False

if convert_sources:
    converter = AnnDataZarrZipConverter(
        chunk_size=1_000_000,
        align_cells=True,
        layer_format="auto",
    )
    store_paths = [converter(path) for path in raw_paths]
else:
    store_paths = raw_paths

store_paths

## Register all datasets

This is the key bulk-register line. `launch(path)` parses metadata only; chunks are opened and decoded later by the databank/prefetch path.

In [ ]:
bank = ScDataBank()

# Key line: path list -> launched datasets -> list[DatasetId]
dataset_ids = bank.register(map(launch, store_paths))

for file_id, did in enumerate(dataset_ids):
    print(
        file_id,
        did,
        "cells=",
        bank.dataset_num_cells(did),
        "genes=",
        bank.dataset_num_genes(did),
        "dtype=",
        bank.dataset_dtype(did),
    )

## Build the `(file_id, cell_id)` dataset

`ScDataLoader` keeps torch's usual sampling/shuffle/batch-size behavior, but the dataset samples must identify cells as two integers: which registered file and which cell in that file. The small `CellIndexDataset` below avoids materializing a huge list of all pairs.

In [ ]:
class CellIndexDataset:
    def __init__(self, cells_per_file):
        self.offsets = [0]
        for count in cells_per_file:
            self.offsets.append(self.offsets[-1] + int(count))

    def __len__(self):
        return self.offsets[-1]

    def __getitem__(self, index):
        if index < 0:
            index += len(self)
        if index < 0 or index >= len(self):
            raise IndexError(index)
        file_id = bisect_right(self.offsets, index) - 1
        cell_id = index - self.offsets[file_id]
        return file_id, cell_id


cells_per_file = [bank.dataset_num_cells(did) for did in dataset_ids]
cell_dataset = CellIndexDataset(cells_per_file)

len(cell_dataset), cell_dataset[0], cell_dataset[len(cell_dataset) - 1]

## Collate decoded `CellBatch` objects

`ScDataLoader` passes a plain dictionary to `collate_fn`. For each `file_id`, `batch["batches"][file_id]` is already decoded by `ScDataBank.prefetch`; `batch["positions"][file_id]` tells where those rows belong in the mixed-file batch order.

In [ ]:
def collate_scdata_batch(batch):
    first = next(iter(batch["batches"].values()))
    x = np.empty((len(batch["cell_ids"]), first.num_genes), dtype=first.data.dtype)

    for file_id, cell_batch in batch["batches"].items():
        positions = batch["positions"][file_id]
        x[positions] = cell_batch.to_numpy()

    return {
        "x": torch.from_numpy(x),
        "file_ids": torch.as_tensor(batch["file_ids"], dtype=torch.long),
        "cell_ids": torch.as_tensor(batch["cell_ids"], dtype=torch.long),
        "gene_names": first.var_names,
    }

## Create the loader

If batches may mix datasets, either ensure all stores have the same gene order or pass an explicit `genes` list. With `missing="zero"`, absent genes are filled with zeros instead of raising.

In [ ]:
genes = None
# genes = ["GeneA", "GeneB", "GeneC"]

loader = ScDataLoader(
    bank,
    cell_dataset,
    batch_size=1024,
    shuffle=True,
    drop_last=False,
    num_workers=0,
    dataset_ids=dataset_ids,
    genes=genes,
    missing="zero",
    collate_fn=collate_scdata_batch,
)

## Consume batches

From this point on, downstream code sees normal torch tensors plus source ids for tracing.

In [ ]:
for step, batch in enumerate(loader):
    x = batch["x"].float()
    file_ids = batch["file_ids"]
    cell_ids = batch["cell_ids"]

    print(step, x.shape, file_ids[:5].tolist(), cell_ids[:5].tolist())

    # model input example:
    # logits = model(x)
    # loss = loss_fn(logits, targets)
    # loss.backward()

    if step == 2:
        break

## Direct access remains available

The same registered ids can be used for direct random access, which is useful for debugging or validation.

In [ ]:
preview = bank.load(dataset_ids[0], [0, 1, 2], genes=genes, missing="zero")
preview.to_numpy().shape, preview.var_names[:5] if preview.var_names is not None else None

## Cleanup

When the bank is no longer needed, unregister ids and close the Rust-backed resource pools.

In [ ]:
bank.unregister_all(dataset_ids)
bank.close()